In [1]:
# day 4 (take 2) - new target variable: delivery delay
# instead of "did this customer churn", we're predicting "will this order arrive late?"
# late = actual delivery date is AFTER the estimated delivery date promised at order time

import pandas as pd
from pathlib import Path

# loading the cleaned data from day 3 (saved as parquet)
PROCESSED_DIR = Path("../data/processed")
orders = pd.read_parquet(PROCESSED_DIR / "orders_with_customer.parquet")

print(f"loaded {len(orders):,} delivered orders")
print()

# the two columns that matter for our new label:
# - order_estimated_delivery_date: the promise olist made to the customer
# - order_delivered_customer_date: when it actually arrived
# if actual > estimated, it's late. that's our label.

# making sure both columns are datetime (should already be from day 2 cleaning)
print("dtype check on the two key columns:")
print(orders[["order_estimated_delivery_date", "order_delivered_customer_date"]].dtypes)
print()

# checking missingness - we already filtered to delivered orders,
# but there were a few weird edge cases with missing delivered_customer_date
print("missingness in key columns:")
print(orders[["order_estimated_delivery_date", "order_delivered_customer_date"]].isnull().sum())

loaded 96,478 delivered orders

dtype check on the two key columns:
order_estimated_delivery_date    datetime64[ns]
order_delivered_customer_date    datetime64[ns]
dtype: object

missingness in key columns:
order_estimated_delivery_date    0
order_delivered_customer_date    8
dtype: int64


In [2]:
# step 2 - compute how late each order was, and define the binary label
# "late" = actual delivered date is AFTER the estimated date promised at order time

# drop the 8 rows with missing delivered_customer_date (can't compute label without it)
print(f"before dropping missing rows: {len(orders):,}")
orders = orders.dropna(subset=["order_delivered_customer_date"]).copy()
print(f"after dropping missing rows:  {len(orders):,}")
print()

# compute delivery delay in days (negative means delivered EARLY, positive means LATE)
# subtracting two datetimes gives a Timedelta. .dt.days extracts the day count as int.
orders["delay_days"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.days

# quick stats on the raw delay
print("distribution of delay_days (negative = early, positive = late):")
print(orders["delay_days"].describe())
print()

# now the binary label: 1 if late (delay > 0), 0 if on time or early
orders["is_late"] = (orders["delay_days"] > 0).astype(int)

# class balance check - the most important number of the day
print("label distribution:")
print(orders["is_late"].value_counts())
print()

late_pct = orders["is_late"].value_counts(normalize=True) * 100
print("label distribution (percentages):")
print(f"  on time (0): {late_pct[0]:.2f}%")
print(f"  late (1):    {late_pct[1]:.2f}%")

before dropping missing rows: 96,478
after dropping missing rows:  96,470

distribution of delay_days (negative = early, positive = late):
count    96470.000000
mean       -11.875889
std         10.182105
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delay_days, dtype: float64

label distribution:
is_late
0    89936
1     6534
Name: count, dtype: int64

label distribution (percentages):
  on time (0): 93.23%
  late (1):    6.77%


In [3]:
# saving the labeled dataset to processed - this is what we'll use for feature engineering tomorrow
# only saving the columns we need: identifiers + the label + the raw delay (for later analysis)

label_df = orders[[
    "order_id",
    "customer_unique_id",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delay_days",
    "is_late",
]].copy()

label_df.to_parquet(PROCESSED_DIR / "delay_labels.parquet", index=False)

print(f"saved delay_labels.parquet: {len(label_df):,} rows")
print(f"columns: {label_df.columns.tolist()}")
print()
print("file size:")
saved = PROCESSED_DIR / "delay_labels.parquet"
print(f"  {saved.name}: {saved.stat().st_size / 1024 / 1024:.2f} MB")
print()
print("first 5 rows:")
print(label_df.head())

saved delay_labels.parquet: 96,470 rows
columns: ['order_id', 'customer_unique_id', 'order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delay_days', 'is_late']

file size:
  delay_labels.parquet: 8.16 MB

first 5 rows:
                           order_id                customer_unique_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  7c396fd4830fd04220f754e42b4e5bff   
1  53cdb2fc8bc7dce0b6741e2150273451  af07308b275d755c9edb36a90c618231   
2  47770eb9100c2d0c44946d9cf07ec65d  3a653a41f6f9fc3d2a113cf8398680e8   
3  949d5b44dbf5de918fe9c16f97b45f8a  7c142cf63193a1473d2e66489a9ae977   
4  ad21c59c0840e6cb83a9ceb5573f8159  72632f0f9dd73dfee390c9b22eb56dd6   

  order_purchase_timestamp order_delivered_customer_date  \
0      2017-10-02 10:56:33           2017-10-10 21:25:13   
1      2018-07-24 20:41:37           2018-08-07 15:27:45   
2      2018-08-08 08:38:49           2018-08-17 18:06:29   
3      2017-11-18 19:28:06           2017-12-02 00:28:42   